## Example 1: Basic Model-to-Schema Auto-generation

In [ ]:
# Example 1: Basic Model-to-Schema Auto-generation
from marshmallow_sqlalchemy import SQLAlchemyAutoSchema, auto_field
from marshmallow import fields
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Boolean, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from datetime import datetime

try:
    print("🔄 Marshmallow SQLAlchemy - Model-to-Schema Auto-generation")
    
    # Create in-memory SQLite database for demo
    engine = create_engine('sqlite:///:memory:', echo=False)
    Base = declarative_base()
    
    # Define SQLAlchemy models
    class User(Base):
        __tablename__ = 'users'
        
        id = Column(Integer, primary_key=True)
        username = Column(String(80), unique=True, nullable=False)
        email = Column(String(120), unique=True, nullable=False)
        full_name = Column(String(200))
        is_active = Column(Boolean, default=True)
        created_at = Column(DateTime, default=datetime.utcnow)
        
        # Relationship
        posts = relationship("Post", back_populates="author")
    
    class Post(Base):
        __tablename__ = 'posts'
        
        id = Column(Integer, primary_key=True)
        title = Column(String(200), nullable=False)
        content = Column(String(2000))
        author_id = Column(Integer, ForeignKey('users.id'), nullable=False)
        created_at = Column(DateTime, default=datetime.utcnow)
        
        # Relationship
        author = relationship("User", back_populates="posts")
    
    # Create tables
    Base.metadata.create_all(engine)
    Session = sessionmaker(bind=engine)
    session = Session()
    
    # Auto-generate schemas from models
    class UserSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = User
            load_instance = True  # Deserialize to model instances
            sqla_session = session
            
        # Override fields if needed
        email = auto_field(dump_only=False)  # Allow email in serialization
        created_at = auto_field(dump_only=True)  # Read-only timestamp
    
    class PostSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = Post
            load_instance = True
            sqla_session = session
            include_relationships = True  # Include related objects
        
        # Custom formatting for timestamps
        created_at = fields.DateTime(dump_only=True, format='%Y-%m-%d %H:%M:%S')
        
        # Nested relationship
        author = fields.Nested(UserSchema, dump_only=True)
    
    # Create sample data
    sample_user = User(
        username='johndoe',
        email='john@example.com',
        full_name='John Doe',
        is_active=True
    )
    
    session.add(sample_user)
    session.commit()
    
    sample_post = Post(
        title='Getting Started with SQLAlchemy',
        content='SQLAlchemy is a powerful Python ORM...',
        author_id=sample_user.id
    )
    
    session.add(sample_post)
    session.commit()
    
    print(f"📊 Created sample data:")
    print(f"   • User: {sample_user.username} ({sample_user.email})")
    print(f"   • Post: {sample_post.title}")
    
    # Test schema serialization
    user_schema = UserSchema()
    post_schema = PostSchema()
    
    # Serialize user
    user_data = user_schema.dump(sample_user)
    print(f"\n🔄 User Serialization:")
    for key, value in user_data.items():
        print(f"   • {key}: {value}")
    
    # Serialize post with nested user
    post_data = post_schema.dump(sample_post)
    print(f"\n📝 Post Serialization (with nested author):")
    for key, value in post_data.items():
        if isinstance(value, dict):
            print(f"   • {key}:")
            for nested_key, nested_value in value.items():
                print(f"       ▸ {nested_key}: {nested_value}")
        else:
            print(f"   • {key}: {value}")
    
    # Test deserialization
    new_user_data = {
        'username': 'janedoe',
        'email': 'jane@example.com',
        'full_name': 'Jane Doe',
        'is_active': True
    }
    
    new_user = user_schema.load(new_user_data)
    session.add(new_user)
    session.commit()
    
    print(f"\n✨ Deserialized new user: {new_user.username}")
    print(f"   • Created instance: {type(new_user).__name__}")
    print(f"   • Database ID: {new_user.id}")
    
    session.close()
    print("✅ Basic model-to-schema auto-generation completed")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("📦 Install with: pip install marshmallow-sqlalchemy")
except Exception as e:
    print(f"⚠️  Demo error: {e}")
    print("🔍 SQLAlchemyAutoSchema auto-generates schemas from models")
    raise AssertionError(
        f"marshmallow_sqlalchemy example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Advanced Field Customization and Validation

In [ ]:
# Example 2: Advanced Field Customization and Validation
from marshmallow_sqlalchemy import SQLAlchemyAutoSchema, auto_field
from marshmallow import fields, validate, validates, validates_schema, ValidationError
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Boolean, Float, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from datetime import datetime
import math
import re

try:
    print("🔧 Marshmallow SQLAlchemy - Advanced Field Customization and Validation")
    
    # Create in-memory SQLite database
    engine = create_engine('sqlite:///:memory:', echo=False)
    Base = declarative_base()
    
    # Define a complex model with various field types
    class Product(Base):
        __tablename__ = 'products'
        
        id = Column(Integer, primary_key=True)
        name = Column(String(100), nullable=False)
        description = Column(Text)
        price = Column(Float, nullable=False)
        stock_quantity = Column(Integer, default=0)
        sku = Column(String(50), unique=True, nullable=False)
        is_active = Column(Boolean, default=True)
        created_at = Column(DateTime, default=datetime.utcnow)
        updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)
    
    # Create tables
    Base.metadata.create_all(engine)
    Session = sessionmaker(bind=engine)
    session = Session()
    
    # Advanced schema with custom validation and field transformations
    class ProductSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = Product
            load_instance = True
            sqla_session = session
            include_fk = True  # Include foreign keys
        
        # Custom field validation and transformation
        name = auto_field(validate=validate.Length(min=3, max=100))
        
        # Price validation with custom error message
        price = auto_field(validate=[
            validate.Range(min=0, error="Price must be non-negative"),
            validate.Range(max=10000, error="Price too high for our system")
        ])
        
        # Stock validation
        stock_quantity = auto_field(validate=validate.Range(min=0, max=10000))
        
        # SKU validation with regex pattern
        sku = auto_field(validate=validate.Regexp(
            r'^[A-Z]{2,3}-\d{4,6}$',
            error="SKU must be in format: XX-1234 or XXX-12345"
        ))
        
        # Read-only timestamps
        created_at = auto_field(dump_only=True, format='%Y-%m-%d %H:%M:%S')
        updated_at = auto_field(dump_only=True, format='%Y-%m-%d %H:%M:%S')
        
        # Virtual field - not in database but calculated
        display_name = fields.Method("get_display_name", dump_only=True)
        
        # Custom field transformation
        formatted_price = fields.Method("get_formatted_price", dump_only=True)
        
        def get_display_name(self, obj):
            """Generate display name with stock info."""
            stock_info = "In Stock" if obj.stock_quantity > 0 else "Out of Stock"
            return f"{obj.name} ({stock_info})"
        
        def get_formatted_price(self, obj):
            """Format price as currency."""
            return f"${obj.price:.2f}"
        
        # Custom field-level validation
        @validates('description')
        def validate_description(self, value, **kwargs):
            if value and len(value.strip()) < 10:
                raise ValidationError("Description must be at least 10 characters long")
            return value.strip() if value else None
        
        # Cross-field validation
        @validates_schema
        def validate_product_data(self, data, **kwargs):
            """Validate relationships between fields."""
            # Check if expensive items have descriptions
            if data.get('price', 0) > 100 and not data.get('description'):
                raise ValidationError("Expensive products (>$100) must have descriptions")
            
            # Check SKU prefix matches name
            name = data.get('name', '')
            sku = data.get('sku', '')
            
            if name and sku and len(name) >= 2:
                expected_prefix = name[:2].upper()
                if not sku.startswith(expected_prefix):
                    raise ValidationError(f"SKU should start with '{expected_prefix}' based on product name")
    
    # Test advanced validation and customization
    schema = ProductSchema()
    
    print(f"\n🧪 Testing Advanced Validation:")
    
    # Test valid product
    valid_product_data = {
        'name': 'Laptop Computer',
        'description': 'High-performance laptop for professional use with fast processor',
        'price': 1299.99,
        'stock_quantity': 15,
        'sku': 'LA-001234',
        'is_active': True
    }
    
    try:
        product = schema.load(valid_product_data)
        session.add(product)
        session.commit()
        
        print(f"   ✅ Valid product created: {product.name}")
        
        # Test serialization with custom fields
        serialized = schema.dump(product)
        print(f"   📊 Serialized data:")
        for key, value in serialized.items():
            print(f"      • {key}: {value}")
        
    except ValidationError as e:
        print(f"   ❌ Validation failed: {e.messages}")
        raise  # the valid product must load; a rejection here is a regression, not part of the demo
    
    # Test validation failures
    print(f"\n🚨 Testing Validation Failures:")
    
    invalid_test_cases = [
        {
            'name': 'TX',  # Too short
            'price': 50,
            'sku': 'TX-1234',
            'description': 'Short'
        },
        {
            'name': 'Invalid SKU Product', 
            'price': -10,  # Negative price
            'sku': 'INVALID',  # Wrong SKU format
            'stock_quantity': -5  # Negative stock
        },
        {
            'name': 'Expensive Product',
            'price': 500,  # Expensive but no description
            'sku': 'EX-5678',
            'stock_quantity': 10
        }
    ]
    
    # The fields each invalid case is rejected on
    expected_error_fields = [{'name', 'description'}, {'price', 'stock_quantity', 'sku'}, {'_schema'}]
    for i, (invalid_data, expected_fields) in enumerate(zip(invalid_test_cases, expected_error_fields, strict=True), 1):
        try:
            schema.load(invalid_data)
        except ValidationError as e:
            assert set(e.messages) == expected_fields, f"Test {i}: rejected on {sorted(e.messages)}, expected {sorted(expected_fields)}"
            print(f"   ✅ Test {i}: Validation correctly failed")
            for field, errors in e.messages.items():
                print(f"      • {field}: {', '.join(errors) if isinstance(errors, list) else errors}")
        else:
            raise AssertionError(f"Test {i}: schema.load accepted invalid data {invalid_data!r}")
    
    # Demonstrate partial updates (useful for PATCH operations)
    print(f"\n🔄 Testing Partial Updates:")
    
    # Create a product first
    base_product = Product(
        name='Smartphone',
        description='Latest smartphone with advanced features',
        price=699.99,
        sku='SM-789012',
        stock_quantity=25
    )
    session.add(base_product)
    session.commit()
    
    # Partial update schema (allows missing fields)
    class ProductUpdateSchema(ProductSchema):
        class Meta(ProductSchema.Meta):
            partial = True  # Allow partial updates
    
    update_schema = ProductUpdateSchema()
    
    # A price PATCH is REJECTED: @validates_schema receives only the PATCH payload, not the instance, so this >$100
    # product looks description-less to the cross-field rule; the instance is left untouched
    try:
        update_schema.load({'price': 649.99, 'stock_quantity': 20}, instance=base_product, partial=True)
    except ValidationError as e:
        assert e.messages == {'_schema': ['Expensive products (>$100) must have descriptions']}, e.messages
        print(f"   ✅ Price PATCH rejected by the cross-field rule: {e.messages['_schema'][0]}")
    else:
        raise AssertionError("a price PATCH of a >$100 product passed the description rule without a description")
    
    # Test partial update: a PATCH that leaves price out passes the rule and updates the existing instance in place
    update_data = {
        'stock_quantity': 20  # Stock adjustment
    }
    update_schema.load(update_data, instance=base_product, partial=True)
    session.commit()
    assert base_product.stock_quantity == 20 and math.isclose(base_product.price, 699.99), (base_product.stock_quantity, base_product.price)
    assert base_product.name == 'Smartphone'
    
    print(f"   ✅ Partial update successful")
    print(f"      • Updated stock: {base_product.stock_quantity}")
    print(f"      • Price unchanged: ${base_product.price}")
    print(f"      • Name unchanged: {base_product.name}")
    
    session.close()
    print("✅ Advanced field customization and validation completed")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("📦 Install with: pip install marshmallow-sqlalchemy")
except Exception as e:
    print(f"⚠️  Demo error: {e}")
    print("🔧 Advanced features show extensive validation and customization")
    raise AssertionError(
        f"marshmallow_sqlalchemy example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Complex Relationships and Nested Schemas

In [ ]:
# Example 3: Complex Relationships and Nested Schemas
from marshmallow_sqlalchemy import SQLAlchemyAutoSchema, auto_field
from marshmallow import fields, post_load
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Boolean, ForeignKey, Table
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship, backref
from datetime import datetime

try:
    print("🔗 Marshmallow SQLAlchemy - Complex Relationships and Nested Schemas")
    
    # Create in-memory SQLite database
    engine = create_engine('sqlite:///:memory:', echo=False)
    Base = declarative_base()
    
    # Many-to-many association table
    user_role_association = Table(
        'user_roles', Base.metadata,
        Column('user_id', Integer, ForeignKey('users.id')),
        Column('role_id', Integer, ForeignKey('roles.id'))
    )
    
    # Define models with complex relationships
    class User(Base):
        __tablename__ = 'users'
        
        id = Column(Integer, primary_key=True)
        username = Column(String(80), unique=True, nullable=False)
        email = Column(String(120), unique=True, nullable=False)
        is_active = Column(Boolean, default=True)
        created_at = Column(DateTime, default=datetime.utcnow)
        
        # One-to-many relationship
        posts = relationship("Post", back_populates="author", lazy='dynamic')
        
        # One-to-one relationship
        profile = relationship("UserProfile", back_populates="user", uselist=False)
        
        # Many-to-many relationship
        roles = relationship("Role", secondary=user_role_association, back_populates="users")
    
    class UserProfile(Base):
        __tablename__ = 'user_profiles'
        
        id = Column(Integer, primary_key=True)
        user_id = Column(Integer, ForeignKey('users.id'), unique=True)
        full_name = Column(String(200))
        bio = Column(String(500))
        location = Column(String(100))
        
        # Back reference to user
        user = relationship("User", back_populates="profile")
    
    class Role(Base):
        __tablename__ = 'roles'
        
        id = Column(Integer, primary_key=True)
        name = Column(String(50), unique=True, nullable=False)
        description = Column(String(200))
        
        # Many-to-many back reference
        users = relationship("User", secondary=user_role_association, back_populates="roles")
    
    class Post(Base):
        __tablename__ = 'posts'
        
        id = Column(Integer, primary_key=True)
        title = Column(String(200), nullable=False)
        content = Column(String(2000))
        author_id = Column(Integer, ForeignKey('users.id'), nullable=False)
        created_at = Column(DateTime, default=datetime.utcnow)
        
        # Many-to-one relationship
        author = relationship("User", back_populates="posts")
        
        # One-to-many relationship
        comments = relationship("Comment", back_populates="post", cascade="all, delete-orphan")
    
    class Comment(Base):
        __tablename__ = 'comments'
        
        id = Column(Integer, primary_key=True)
        content = Column(String(500), nullable=False)
        post_id = Column(Integer, ForeignKey('posts.id'), nullable=False)
        author_id = Column(Integer, ForeignKey('users.id'), nullable=False)
        created_at = Column(DateTime, default=datetime.utcnow)
        
        # Relationships
        post = relationship("Post", back_populates="comments")
        author = relationship("User")
    
    # Create all tables
    Base.metadata.create_all(engine)
    Session = sessionmaker(bind=engine)
    session = Session()
    
    # Define nested schemas with different relationship handling strategies
    class UserProfileSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = UserProfile
            load_instance = True
            sqla_session = session
        
        # Format timestamp
        created_at = fields.DateTime(dump_only=True, format='%Y-%m-%d %H:%M:%S')
    
    class RoleSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = Role
            load_instance = True
            sqla_session = session
        
        # Count how many users have this role
        user_count = fields.Method("get_user_count", dump_only=True)
        
        def get_user_count(self, obj):
            return len(obj.users)
    
    class CommentSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = Comment
            load_instance = True
            sqla_session = session
            include_relationships = False  # Don't auto-include to avoid circular refs
        
        # Include author username only
        author_username = fields.Method("get_author_username", dump_only=True)
        created_at = fields.DateTime(dump_only=True, format='%Y-%m-%d %H:%M:%S')
        
        def get_author_username(self, obj):
            return obj.author.username if obj.author else None
    
    class PostSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = Post
            load_instance = True
            sqla_session = session
            include_relationships = False  # Manual control over relationships
        
        # Nested relationships with different strategies
        author = fields.Nested(lambda: UserSchema(only=['id', 'username', 'email']), dump_only=True)
        comments = fields.Nested(CommentSchema, many=True, dump_only=True)
        
        # Virtual fields
        comment_count = fields.Method("get_comment_count", dump_only=True)
        created_at = fields.DateTime(dump_only=True, format='%Y-%m-%d %H:%M:%S')
        
        def get_comment_count(self, obj):
            return len(obj.comments)
    
    class UserSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = User
            load_instance = True
            sqla_session = session
            include_relationships = False  # Manual relationship control
        
        # Nested one-to-one relationship
        profile = fields.Nested(UserProfileSchema, dump_only=True)
        
        # Nested many-to-many relationship
        roles = fields.Nested(RoleSchema, many=True, dump_only=True)
        
        # Nested one-to-many with limited fields to avoid deep nesting
        posts = fields.Nested(lambda: PostSchema(only=['id', 'title', 'created_at', 'comment_count']), 
                             many=True, dump_only=True)
        
        # Virtual fields
        post_count = fields.Method("get_post_count", dump_only=True)
        role_names = fields.Method("get_role_names", dump_only=True)
        created_at = fields.DateTime(dump_only=True, format='%Y-%m-%d %H:%M:%S')
        
        def get_post_count(self, obj):
            return obj.posts.count() if hasattr(obj.posts, 'count') else len(obj.posts)
        
        def get_role_names(self, obj):
            return [role.name for role in obj.roles]
    
    # Create test data with complex relationships
    print(f"\n🏗️  Creating Complex Test Data:")
    
    # Create roles
    admin_role = Role(name='admin', description='Administrator role')
    user_role = Role(name='user', description='Regular user role')
    moderator_role = Role(name='moderator', description='Content moderator role')
    
    session.add_all([admin_role, user_role, moderator_role])
    session.commit()
    
    # Create users with profiles and roles
    admin_user = User(username='admin', email='admin@example.com')
    admin_profile = UserProfile(
        user=admin_user, 
        full_name='Admin User', 
        bio='System administrator', 
        location='Server Room'
    )
    admin_user.roles = [admin_role, moderator_role]
    
    regular_user = User(username='johndoe', email='john@example.com')
    user_profile = UserProfile(
        user=regular_user, 
        full_name='John Doe', 
        bio='Regular user who likes to post', 
        location='New York'
    )
    regular_user.roles = [user_role]
    
    session.add_all([admin_user, admin_profile, regular_user, user_profile])
    session.commit()
    
    # Create posts with comments
    post1 = Post(
        title='Welcome to the Platform',
        content='This is our first post welcoming everyone to the platform...',
        author=admin_user
    )
    
    post2 = Post(
        title='Getting Started Guide',
        content='Here\'s how to get started with using our platform effectively...',
        author=regular_user
    )
    
    session.add_all([post1, post2])
    session.commit()
    
    # Create comments
    comment1 = Comment(
        content='Great post! Looking forward to more content.',
        post=post1,
        author=regular_user
    )
    
    comment2 = Comment(
        content='Very helpful guide, thank you!',
        post=post2,
        author=admin_user
    )
    
    comment3 = Comment(
        content='I have a question about this section...',
        post=post2,
        author=regular_user
    )
    
    session.add_all([comment1, comment2, comment3])
    session.commit()
    
    print(f"   ✅ Created users: {admin_user.username}, {regular_user.username}")
    print(f"   ✅ Created posts: {len([post1, post2])} posts")
    print(f"   ✅ Created comments: {len([comment1, comment2, comment3])} comments")
    
    # Test nested serialization
    print(f"\n🔄 Testing Nested Serialization:")
    
    user_schema = UserSchema()
    post_schema = PostSchema()
    
    # Serialize admin user with all nested relationships
    admin_data = user_schema.dump(admin_user)
    
    print(f"\n👤 Admin User with Nested Data:")
    print(f"   • Username: {admin_data['username']}")
    print(f"   • Email: {admin_data['email']}")
    print(f"   • Post count: {admin_data['post_count']}")
    print(f"   • Role names: {', '.join(admin_data['role_names'])}")
    
    print(f"   📋 Profile:")
    profile_data = admin_data.get('profile', {})
    for key, value in profile_data.items():
        if key != 'user_id':  # Skip foreign key
            print(f"      • {key}: {value}")
    
    print(f"   🏷️  Roles:")
    for role in admin_data.get('roles', []):
        print(f"      • {role['name']}: {role['description']} ({role['user_count']} users)")
    
    # Serialize post with nested relationships
    print(f"\n📝 Post with Nested Data:")
    post_data = post_schema.dump(post2)
    
    print(f"   • Title: {post_data['title']}")
    print(f"   • Author: {post_data['author']['username']}")
    print(f"   • Comment count: {post_data['comment_count']}")
    print(f"   • Created: {post_data['created_at']}")
    
    print(f"   💬 Comments:")
    for comment in post_data.get('comments', []):
        print(f"      • {comment['author_username']}: {comment['content'][:30]}...")
    
    session.close()
    print("✅ Complex relationships and nested schemas completed")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("📦 Install with: pip install marshmallow-sqlalchemy")
except Exception as e:
    print(f"⚠️  Demo error: {e}")
    print("🔗 Complex relationships demonstrate advanced ORM integration")
    raise AssertionError(
        f"marshmallow_sqlalchemy example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: API Integration and Performance Optimization

In [ ]:
# Example 4: API Integration and Performance Optimization
from marshmallow_sqlalchemy import SQLAlchemyAutoSchema, auto_field
from marshmallow import fields, pre_load, post_dump, ValidationError
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Boolean, event
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, scoped_session, Query
from sqlalchemy.orm.strategy_options import Load
from datetime import datetime
from typing import Dict, Any, List

try:
    print("🚀 Marshmallow SQLAlchemy - API Integration and Performance Optimization")
    
    # Create in-memory SQLite database with connection pooling
    engine = create_engine(
        'sqlite:///:memory:', 
        echo=False,
        pool_pre_ping=True,  # Verify connections before use
        connect_args={'check_same_thread': False}
    )
    Base = declarative_base()
    
    # Define optimized model
    class APIResource(Base):
        __tablename__ = 'api_resources'
        
        id = Column(Integer, primary_key=True)
        name = Column(String(100), nullable=False, index=True)  # Indexed for performance
        description = Column(String(500))
        status = Column(String(20), default='active', index=True)  # Indexed for filtering
        priority = Column(Integer, default=1)
        resource_metadata = Column(String(1000))  # Renamed from 'metadata' to avoid reserved name
        created_at = Column(DateTime, default=datetime.utcnow, index=True)
        updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)
        
        def __repr__(self):
            return f"<APIResource(id={self.id}, name='{self.name}', status='{self.status}')>"
    
    # Create tables
    Base.metadata.create_all(engine)
    
    # Use scoped session for thread safety in web applications
    session_factory = sessionmaker(bind=engine)
    Session = scoped_session(session_factory)
    
    # Performance-optimized schema with various API patterns
    class APIResourceSchema(SQLAlchemyAutoSchema):
        class Meta:
            model = APIResource
            load_instance = True
            sqla_session = Session
        
        # Custom serialization for different API contexts
        id = auto_field(dump_only=True)
        created_at = fields.DateTime(dump_only=True, format='iso')
        updated_at = fields.DateTime(dump_only=True, format='iso')
        
        # Computed field for API responses
        resource_url = fields.Method("get_resource_url", dump_only=True)
        
        # Transform resource_metadata JSON string for API
        metadata_dict = fields.Method("get_metadata_dict", "set_metadata_dict")
        
        def get_resource_url(self, obj) -> str:
            """Generate API URL for this resource."""
            return f"/api/v1/resources/{obj.id}"
        
        def get_metadata_dict(self, obj) -> Dict[str, Any]:
            """Parse resource_metadata string as dictionary."""
            try:
                import json
                return json.loads(obj.resource_metadata) if obj.resource_metadata else {}
            except (json.JSONDecodeError, TypeError):
                return {}
        
        def set_metadata_dict(self, value) -> str:
            """Convert dictionary to JSON string for storage."""
            try:
                import json
                return json.dumps(value) if value else None
            except TypeError:
                return None
        
        # Pre-processing for API input normalization
        @pre_load
        def normalize_input(self, data, **kwargs):
            """Normalize and validate API input."""
            if isinstance(data, dict):
                # Normalize status values
                if 'status' in data:
                    data['status'] = data['status'].lower().strip()
                
                # Ensure priority is within bounds
                if 'priority' in data:
                    data['priority'] = max(1, min(10, int(data.get('priority', 1))))
                
                # Clean up string fields
                for field in ['name', 'description']:
                    if field in data and isinstance(data[field], str):
                        data[field] = data[field].strip()
            
            return data
        
        # Post-processing for API output optimization
        @post_dump
        def finalize_output(self, data, **kwargs):
            """Optimize output for API responses."""
            # Add computed fields
            data['display_name'] = f"{data.get('name', 'Unknown')} ({data.get('status', 'unknown')})"
            
            # Add API metadata
            data['_links'] = {
                'self': data.get('resource_url', ''),
                'update': f"{data.get('resource_url', '')}/update",
                'delete': f"{data.get('resource_url', '')}/delete"
            }
            
            # Remove internal fields if needed
            if 'resource_metadata' in data and 'metadata_dict' in data:
                del data['resource_metadata']  # Remove raw metadata, keep parsed version
            
            return data
    
    # Different schema variants for different API endpoints
    class APIResourceListSchema(APIResourceSchema):
        """Optimized schema for list endpoints - minimal fields."""
        class Meta(APIResourceSchema.Meta):
            # display_name is added by @post_dump and is not a model field, so it is not listed in Meta.fields.
            fields = ('id', 'name', 'status', 'priority', 'resource_url')
    
    class APIResourceDetailSchema(APIResourceSchema):
        """Full schema for detail endpoints - all fields."""
        pass
    
    class APIResourceCreateSchema(APIResourceSchema):
        """Schema for creation - no read-only fields."""
        class Meta(APIResourceSchema.Meta):
            exclude = ('id', 'created_at', 'updated_at', 'resource_url')
    
    # Bulk operations support
    class BulkAPIManager:
        """Manager for bulk API operations with performance optimization."""
        
        def __init__(self, session):
            self.session = session
        
        def bulk_create(self, data_list: List[Dict], schema: APIResourceSchema) -> Dict[str, Any]:
            """Efficiently create multiple resources."""
            results = {
                'created': [],
                'errors': [],
                'total_processed': len(data_list)
            }
            
            # Validate all data first
            validated_items = []
            for i, item_data in enumerate(data_list):
                try:
                    validated_item = schema.load(item_data)
                    validated_items.append(validated_item)
                except ValidationError as e:
                    results['errors'].append({
                        'index': i,
                        'data': item_data,
                        'errors': e.messages
                    })
            
            if validated_items:
                # Bulk insert for performance
                try:
                    self.session.add_all(validated_items)
                    self.session.commit()
                    
                    # Serialize created items
                    for item in validated_items:
                        results['created'].append(schema.dump(item))
                        
                except Exception as e:
                    self.session.rollback()
                    results['errors'].append({
                        'type': 'database_error',
                        'message': str(e)
                    })
            
            return results
        
        def bulk_update(self, updates: List[Dict], schema: APIResourceSchema) -> Dict[str, Any]:
            """Efficiently update multiple resources."""
            results = {
                'updated': [],
                'errors': [],
                'total_processed': len(updates)
            }
            
            for update_data in updates:
                try:
                    resource_id = update_data.pop('id', None)
                    if not resource_id:
                        results['errors'].append({
                            'data': update_data,
                            'error': 'ID required for updates'
                        })
                        continue
                    
                    # Find and update resource
                    resource = self.session.query(APIResource).get(resource_id)
                    if not resource:
                        results['errors'].append({
                            'id': resource_id,
                            'error': 'Resource not found'
                        })
                        continue
                    
                    # Apply partial update
                    updated_resource = schema.load(update_data, instance=resource, partial=True)
                    results['updated'].append(schema.dump(updated_resource))
                    
                except ValidationError as e:
                    results['errors'].append({
                        'id': resource_id,
                        'errors': e.messages
                    })
            
            try:
                self.session.commit()
            except Exception as e:
                self.session.rollback()
                results['errors'].append({
                    'type': 'database_error',
                    'message': str(e)
                })
            
            return results
    
    # Create test data and demonstrate API patterns
    print(f"\n🏗️  Creating Test Data for API Demonstration:")
    
    session = Session()
    
    # Sample API resource data
    sample_resources = [
        {
            'name': 'User Authentication API',
            'description': 'Handles user login and authentication',
            'status': 'active',
            'priority': 5,
            'metadata_dict': {'version': '2.1', 'rate_limit': 1000}
        },
        {
            'name': 'Data Processing Pipeline',
            'description': 'Processes incoming data streams',
            'status': 'maintenance',
            'priority': 8,
            'metadata_dict': {'queue_size': 500, 'workers': 4}
        },
        {
            'name': 'Notification Service',
            'description': 'Sends email and SMS notifications',
            'status': 'active',
            'priority': 3,
            'metadata_dict': {'providers': ['smtp', 'twilio'], 'retry_count': 3}
        }
    ]
    
    # Test API schemas with bulk operations
    bulk_manager = BulkAPIManager(session)
    create_schema = APIResourceCreateSchema()
    list_schema = APIResourceListSchema()
    detail_schema = APIResourceDetailSchema()
    
    # Bulk create demonstration
    print(f"\n📦 Testing Bulk Create:")
    create_results = bulk_manager.bulk_create(sample_resources, create_schema)
    
    print(f"   ✅ Created: {len(create_results['created'])} resources")
    print(f"   ❌ Errors: {len(create_results['errors'])} errors")
    # every sample resource is valid for the create schema, so any bulk-create error is a failure
    assert not create_results['errors'], create_results['errors']
    
    # The create schema excludes read-only fields such as resource_url, so it is read with .get().
    for resource in create_results['created'][:2]:  # Show first 2
        print(f"      • {resource['display_name']} - "
              f"{resource.get('resource_url', '(excluded by the create schema)')}")
    
    # Test different schema outputs
    print(f"\n📋 Testing Different API Schema Outputs:")
    
    # List all resources (minimal schema)
    resources = session.query(APIResource).all()
    
    print(f"   📄 List Schema Output:")
    list_data = list_schema.dump(resources, many=True)
    for item in list_data:
        print(f"      • ID {item['id']}: {item['display_name']}")
    
    # Detail view (full schema)
    if resources:
        detail_resource = resources[0]
        detail_data = detail_schema.dump(detail_resource)
        
        print(f"\n   🔍 Detail Schema Output:")
        for key, value in detail_data.items():
            if key not in ['_links']:  # Skip complex nested data for brevity
                print(f"      • {key}: {value}")
        
        print(f"      • API Links: {len(detail_data.get('_links', {}))} endpoints")
    
    # Performance optimization demonstration
    print(f"\n⚡ Performance Optimization Features:")
    
    performance_features = [
        "Bulk insert/update operations for high-throughput APIs",
        "Scoped sessions for thread safety in web applications",
        "Database connection pooling and pre-ping validation",
        "Indexed fields for efficient querying and filtering", 
        "Schema variants for different API endpoints (list vs detail)",
        "Input normalization and output optimization hooks",
        "Lazy loading strategies for nested relationships",
        "Error handling and validation with detailed feedback"
    ]
    
    for i, feature in enumerate(performance_features, 1):
        print(f"   {i}. {feature}")
    
    # API integration patterns
    print(f"\n🌐 API Integration Patterns:")
    
    api_patterns = {
        "RESTful Resource URLs": "Auto-generated resource URLs for HATEOAS",
        "Bulk Operations": "Efficient batch create/update/delete endpoints",
        "Schema Variants": "Different schemas for list/detail/create operations",
        "Input Normalization": "Automatic data cleaning and validation",
        "Error Handling": "Structured error responses with field-level details",
        "Hypermedia Links": "Self-describing API responses with navigation links",
        "Performance Optimization": "Bulk operations and connection pooling",
        "Thread Safety": "Scoped sessions for concurrent request handling"
    }
    
    for pattern, description in api_patterns.items():
        print(f"   • {pattern}: {description}")
    
    # Clean up
    session.close()
    Session.remove()  # Important for scoped sessions
    
    print("✅ API integration and performance optimization completed")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("📦 Install with: pip install marshmallow-sqlalchemy")
except Exception as e:
    print(f"⚠️  Demo error: {e}")
    print("🚀 API integration demonstrates production-ready patterns")
    raise AssertionError(
        f"marshmallow_sqlalchemy example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Production Deployment and Enterprise Patterns

In [ ]:
# Example 5: Production Deployment and Enterprise Patterns
from marshmallow_sqlalchemy import SQLAlchemyAutoSchema, auto_field
from marshmallow import fields, Schema, validate, post_load, ValidationError, EXCLUDE
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Boolean, event
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, scoped_session
from sqlalchemy.pool import QueuePool
from datetime import datetime, timedelta
from typing import Dict, Any, Optional, List
import logging
import threading
import time

try:
    print("🏭 Marshmallow SQLAlchemy - Production Deployment and Enterprise Patterns")
    
    # Configure logging for production monitoring
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    
    # Production-grade database configuration
    class DatabaseConfig:
        """Production database configuration with connection pooling."""
        
        def __init__(self):
            self.engine = create_engine(
                'sqlite:///:memory:',  # In production: use PostgreSQL/MySQL
                echo=False,  # Disable SQL logging in production
                pool_size=20,  # Connection pool size
                max_overflow=30,  # Additional connections beyond pool_size
                pool_pre_ping=True,  # Verify connections are alive
                pool_recycle=3600,  # Recycle connections every hour
                poolclass=QueuePool,  # Use queue-based connection pool
                connect_args={'check_same_thread': False}
            )
            
            # Event listeners for monitoring
            event.listen(self.engine, 'connect', self._on_connect)
            event.listen(self.engine, 'checkout', self._on_checkout)
            
            self.session_factory = sessionmaker(
                bind=self.engine,
                expire_on_commit=False,  # Keep objects after commit
                autoflush=True,  # Auto-flush before queries
                autocommit=False  # Explicit transaction control
            )
            
            self.Session = scoped_session(self.session_factory)
            
            # Connection monitoring
            self.connection_stats = {
                'total_connections': 0,
                'active_connections': 0,
                'last_checkout': None
            }
        
        def _on_connect(self, dbapi_conn, connection_record):
            """Monitor database connections."""
            self.connection_stats['total_connections'] += 1
            logger.info(f"Database connection established. Total: {self.connection_stats['total_connections']}")
        
        def _on_checkout(self, dbapi_conn, connection_record, connection_proxy):
            """Monitor connection checkouts."""
            self.connection_stats['active_connections'] += 1
            self.connection_stats['last_checkout'] = datetime.utcnow()
    
    # Enterprise data model with audit trail
    Base = declarative_base()
    
    class AuditMixin:
        """Mixin for audit trail functionality."""
        created_at = Column(DateTime, default=datetime.utcnow, nullable=False, index=True)
        updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow, nullable=False)
        created_by = Column(String(100))  # User who created the record
        updated_by = Column(String(100))  # User who last updated the record
        version = Column(Integer, default=1)  # Optimistic locking version
        is_deleted = Column(Boolean, default=False, index=True)  # Soft delete flag
    
    class EnterpriseEntity(Base, AuditMixin):
        """Production entity with full audit capabilities."""
        __tablename__ = 'enterprise_entities'
        
        id = Column(Integer, primary_key=True)
        name = Column(String(200), nullable=False, index=True)
        category = Column(String(50), nullable=False, index=True)
        status = Column(String(20), default='active', index=True)
        priority = Column(Integer, default=1, index=True)
        business_rules = Column(String(2000))  # JSON-encoded business rules
        compliance_data = Column(String(1000))  # Compliance-related data
        
        def __repr__(self):
            return f"<EnterpriseEntity(id={self.id}, name='{self.name}', status='{self.status}')>"
    
    # Initialize production database
    db_config = DatabaseConfig()
    Base.metadata.create_all(db_config.engine)
    
    # Production-grade schema with enterprise features
    class EnterpriseEntitySchema(SQLAlchemyAutoSchema):
        """Enterprise schema with validation, security, and audit features."""
        
        class Meta:
            model = EnterpriseEntity
            load_instance = True
            sqla_session = db_config.Session
            unknown = EXCLUDE  # Security: ignore unknown fields
        
        # Input validation with business rules
        name = auto_field(validate=[
            validate.Length(min=3, max=200),
            validate.Regexp(r'^[a-zA-Z0-9\s\-_\.]+$', error="Name contains invalid characters")
        ])
        
        category = auto_field(validate=validate.OneOf([
            'product', 'service', 'process', 'policy', 'system'
        ]))
        
        status = auto_field(validate=validate.OneOf([
            'active', 'inactive', 'pending', 'archived', 'suspended'
        ]))
        
        priority = auto_field(validate=validate.Range(min=1, max=10))
        
        # Audit fields - read-only for API consumers
        id = auto_field(dump_only=True)
        created_at = fields.DateTime(dump_only=True, format='iso')
        updated_at = fields.DateTime(dump_only=True, format='iso')
        created_by = auto_field(dump_only=True)
        updated_by = auto_field(dump_only=True)
        version = auto_field(dump_only=True)
        
        # Computed fields for enterprise reporting
        age_in_days = fields.Method("get_age_in_days", dump_only=True)
        compliance_status = fields.Method("get_compliance_status", dump_only=True)
        risk_level = fields.Method("get_risk_level", dump_only=True)
        
        def get_age_in_days(self, obj) -> int:
            """Calculate entity age for reporting."""
            if obj.created_at:
                delta = datetime.utcnow() - obj.created_at
                return delta.days
            return 0
        
        def get_compliance_status(self, obj) -> str:
            """Determine compliance status based on business rules."""
            try:
                import json
                compliance = json.loads(obj.compliance_data) if obj.compliance_data else {}
                last_audit = compliance.get('last_audit_date')
                
                if not last_audit:
                    return 'non_compliant'
                
                # Simple compliance check (in production: more complex logic)
                audit_date = datetime.fromisoformat(last_audit.replace('Z', '+00:00'))
                days_since_audit = (datetime.utcnow() - audit_date.replace(tzinfo=None)).days
                
                if days_since_audit <= 90:
                    return 'compliant'
                elif days_since_audit <= 180:
                    return 'warning'
                else:
                    return 'non_compliant'
                    
            except (json.JSONDecodeError, ValueError, TypeError):
                return 'unknown'
        
        def get_risk_level(self, obj) -> str:
            """Calculate risk level based on multiple factors."""
            risk_score = 0
            
            # Age factor
            age = self.get_age_in_days(obj)
            if age > 365:
                risk_score += 2
            elif age > 180:
                risk_score += 1
            
            # Priority factor
            if obj.priority >= 8:
                risk_score += 3
            elif obj.priority >= 5:
                risk_score += 1
            
            # Status factor
            if obj.status in ['suspended', 'archived']:
                risk_score += 2
            elif obj.status == 'inactive':
                risk_score += 1
            
            # Compliance factor
            compliance_status = self.get_compliance_status(obj)
            if compliance_status == 'non_compliant':
                risk_score += 3
            elif compliance_status == 'warning':
                risk_score += 1
            
            # Return risk level
            if risk_score >= 6:
                return 'high'
            elif risk_score >= 3:
                return 'medium'
            else:
                return 'low'
    
    # Enterprise API service with comprehensive patterns
    class EnterpriseAPIService:
        """Production API service with enterprise patterns."""
        
        def __init__(self, db_config: DatabaseConfig):
            self.db_config = db_config
            self.schema = EnterpriseEntitySchema()
            self.request_count = 0
            self.error_count = 0
            self.start_time = datetime.utcnow()
        
        def create_entity(self, data: Dict, user_context: Dict) -> Dict[str, Any]:
            """Create entity with full audit trail."""
            session = self.db_config.Session()
            
            try:
                # Add audit information
                data['created_by'] = user_context.get('username', 'system')
                data['updated_by'] = user_context.get('username', 'system')
                
                # Load and validate
                entity = self.schema.load(data)
                
                # Business rule validation
                self._validate_business_rules(entity, user_context)
                
                session.add(entity)
                session.commit()
                
                result = self.schema.dump(entity)
                self.request_count += 1
                
                logger.info(f"Entity created: {entity.id} by {user_context.get('username')}")
                
                return {
                    'success': True,
                    'data': result,
                    'message': 'Entity created successfully'
                }
                
            except ValidationError as e:
                session.rollback()
                self.error_count += 1
                logger.warning(f"Validation error: {e.messages}")
                
                return {
                    'success': False,
                    'errors': e.messages,
                    'message': 'Validation failed'
                }
                
            except Exception as e:
                session.rollback()
                self.error_count += 1
                logger.error(f"Unexpected error: {str(e)}")
                
                return {
                    'success': False,
                    'error': 'Internal server error',
                    'message': 'An unexpected error occurred'
                }
                
            finally:
                session.close()
        
        def _validate_business_rules(self, entity: EnterpriseEntity, user_context: Dict):
            """Apply enterprise business rules validation."""
            # Example business rules
            if entity.priority >= 8 and entity.status != 'active':
                raise ValidationError("High priority entities must be active")
            
            # User permission check
            user_role = user_context.get('role', 'user')
            if entity.category == 'system' and user_role not in ['admin', 'system_admin']:
                raise ValidationError("Insufficient permissions to create system entities")
            
            # Compliance requirement
            if entity.category in ['policy', 'process'] and not entity.compliance_data:
                raise ValidationError("Policy and process entities require compliance data")
        
        def get_health_status(self) -> Dict[str, Any]:
            """Get service health and performance metrics."""
            uptime = datetime.utcnow() - self.start_time
            
            return {
                'status': 'healthy',
                'uptime_seconds': uptime.total_seconds(),
                'requests_processed': self.request_count,
                'error_count': self.error_count,
                'error_rate': self.error_count / max(self.request_count, 1),
                'database': {
                    'total_connections': self.db_config.connection_stats['total_connections'],
                    'active_connections': self.db_config.connection_stats['active_connections'],
                    'last_checkout': self.db_config.connection_stats['last_checkout'].isoformat() if self.db_config.connection_stats['last_checkout'] else None
                },
                'memory_usage': 'Not implemented in demo',
                'version': '1.0.0'
            }
    
    # Demonstrate enterprise patterns
    print(f"\n🏗️  Demonstrating Enterprise Production Patterns:")
    
    # Initialize API service
    api_service = EnterpriseAPIService(db_config)
    
    # Sample enterprise entities
    enterprise_entities = [
        {
            'name': 'Customer Data Processing Policy',
            'category': 'policy',
            'status': 'active',
            'priority': 9,
            'business_rules': '{"data_retention": 365, "encryption_required": true}',
            'compliance_data': '{"last_audit_date": "2024-01-15", "auditor": "ComplianceCorp", "status": "passed"}'
        },
        {
            'name': 'Payment Gateway Integration',
            'category': 'system',
            'status': 'active',
            'priority': 8,
            'business_rules': '{"timeout": 30, "retry_attempts": 3}',
            'compliance_data': '{"pci_compliant": true, "last_scan": "2024-02-01"}'
        },
        {
            'name': 'Marketing Campaign Service',
            'category': 'service',
            'status': 'pending',
            'priority': 5,
            'business_rules': '{"max_recipients": 10000, "frequency_limit": "daily"}',
            'compliance_data': '{"gdpr_compliant": true, "opt_out_mechanism": true}'
        }
    ]
    
    # Create entities with different user contexts
    user_contexts = [
        {'username': 'admin_user', 'role': 'admin'},
        {'username': 'system_admin', 'role': 'system_admin'},
        {'username': 'regular_user', 'role': 'user'}
    ]
    
    print(f"\n📦 Creating Enterprise Entities:")
    created_entities = []
    
    for i, entity_data in enumerate(enterprise_entities):
        user_context = user_contexts[i % len(user_contexts)]
        result = api_service.create_entity(entity_data, user_context)
        
        if result['success']:
            entity = result['data']
            created_entities.append(entity)
            print(f"   ✅ Created: {entity['name']} (Risk: {entity['risk_level']})")
            print(f"      • Compliance: {entity['compliance_status']}")
            print(f"      • Created by: {entity['created_by']}")
        else:
            print(f"   ❌ Failed: {result.get('message', 'Unknown error')}")
            if 'errors' in result:
                # a field error carries {field: [messages]}; a business-rule ValidationError carries a bare list
                errors_by_field = result['errors'] if isinstance(result['errors'], dict) else {'_schema': result['errors']}
                for field, errors in errors_by_field.items():
                    print(f"      • {field}: {', '.join(errors) if isinstance(errors, list) else errors}")
            # All three entities satisfy the business rules for the user creating them (the policy carries compliance
            # data, the system entity is created by a system_admin), so a rejection is a regression, not part of the demo
            raise AssertionError(f"create_entity rejected {entity_data['name']!r}: {result}")
    
    # Health monitoring demonstration
    print(f"\n📊 Service Health Monitoring:")
    health_status = api_service.get_health_status()
    
    for key, value in health_status.items():
        if isinstance(value, dict):
            print(f"   • {key}:")
            for sub_key, sub_value in value.items():
                print(f"      ▸ {sub_key}: {sub_value}")
        else:
            print(f"   • {key}: {value}")
    
    # Production readiness checklist
    print(f"\n✅ Production Readiness Features:")
    
    production_features = [
        "Connection pooling with monitoring and health checks",
        "Comprehensive audit trail with user context tracking",
        "Business rules validation and compliance checking", 
        "Enterprise security patterns (input validation, authorization)",
        "Error handling with logging and monitoring integration",
        "Performance metrics and health status endpoints",
        "Soft delete functionality for data retention requirements",
        "Optimistic locking for concurrent update protection",
        "Configurable validation rules for different environments",
        "Thread-safe session management with scoped sessions"
    ]
    
    for i, feature in enumerate(production_features, 1):
        print(f"   {i:2d}. {feature}")
    
    # Deployment considerations
    print(f"\n🚀 Enterprise Deployment Considerations:")
    
    deployment_patterns = {
        "Database": "PostgreSQL/MySQL with read replicas and connection pooling",
        "Caching": "Redis for session management and frequently accessed data",
        "Load Balancing": "HAProxy/Nginx with health checks and failover",
        "Monitoring": "Prometheus/Grafana for metrics, ELK stack for logging",
        "Security": "OAuth 2.0/JWT authentication, input sanitization, HTTPS",
        "Scalability": "Horizontal scaling with stateless services",
        "Backup": "Automated backups with point-in-time recovery",
        "Compliance": "Audit logging, data retention policies, GDPR compliance",
        "Testing": "Unit tests, integration tests, load testing",
        "Documentation": "API documentation, runbooks, deployment guides"
    }
    
    for area, description in deployment_patterns.items():
        print(f"   • {area}: {description}")
    
    # Clean up resources
    db_config.Session.remove()
    
    print("✅ Production deployment and enterprise patterns completed")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("📦 Install with: pip install marshmallow-sqlalchemy")
except Exception as e:
    print(f"⚠️  Demo error: {e}")
    print("🏭 Enterprise patterns demonstrate production-ready architecture")
    raise AssertionError(
        f"marshmallow_sqlalchemy example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e